In [ ]:
# =============================================================================
#  Stratified Analysis by Tc Range
#  Inputs : Final_dataset_83.csv
#           UCI_family_subFamily.csv
#  Output : tc_stratified_results.csv
# =============================================================================

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import StackingRegressor, RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 1. LOAD DATA
# =============================================================================
df  = pd.read_csv('Final_dataset_83.csv')
uci = pd.read_csv('UCI_family_subFamily.csv')
# print(df.columns)
# print(uci.columns)


df['family']    = uci['family'].values
df['subfamily'] = uci['subfamily'].values

drop_cols = ['critical_temp', 'subfamily', 'family', 'low_confidence', 'material',
       'is_ambiguous', 'vF_predicted_log10',
       'prediction_source']
feat_cols = [c for c in df.columns if c not in drop_cols]

X = df[feat_cols].values
y = df['critical_temp'].values

print(f"Dataset : {X.shape[0]:,} samples  |  {X.shape[1]} features")
print(f"Tc range: {y.min():.2f} – {y.max():.2f} K\n")

# =============================================================================
# 2. SCALING
# =============================================================================
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# =============================================================================
# 3. STACKING ENSEMBLE — OOF PREDICTIONS (10-fold CV)
#    Using cross_val_predict gives unbiased OOF predictions for every sample,
#    which is the cleanest way to stratify without a fixed train/test split.
# =============================================================================
MODEL_DIR = "models_83/tuning_50_HPO"

lasso_best = joblib.load(f"{MODEL_DIR}/lasso_best.pkl")
knn_best   = joblib.load(f"{MODEL_DIR}/knn_best.pkl")
svr_best   = joblib.load(f"{MODEL_DIR}/svr_best.pkl")
mlp_best   = joblib.load(f"{MODEL_DIR}/mlp_best.pkl")
best_rf    = joblib.load(f"{MODEL_DIR}/best_rf.pkl")
best_lgb   = joblib.load(f"{MODEL_DIR}/best_lgb.pkl")

base_models = [
    ("knn",  knn_best),
    ("svr",  svr_best),
    ("mlp",  mlp_best),
    ("lgbm", best_lgb),
    ("rf",   best_rf),
]
meta = Ridge(alpha=2.0)

stack = StackingRegressor(
    estimators=base_models,
    final_estimator=meta,
    cv=5,
    n_jobs=-1,
    passthrough=False
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("Running 5-fold OOF predictions (this will take a few minutes)...")
oof_pred = cross_val_predict(stack, X_scaled, y, cv=kf, n_jobs=1)
oof_pred = np.clip(oof_pred, 0, None)
print("Done.\n")

residuals = oof_pred - y

# =============================================================================
# 4. OVERALL METRICS
# =============================================================================
r2_overall   = r2_score(y, oof_pred)
rmse_overall = np.sqrt(mean_squared_error(y, oof_pred))
mae_overall  = mean_absolute_error(y, oof_pred)

print("=" * 55)
print("OVERALL OOF PERFORMANCE")
print(f"  R²   : {r2_overall:.4f}")
print(f"  RMSE : {rmse_overall:.4f} K")
print(f"  MAE  : {mae_overall:.4f} K")
print("=" * 55)

# =============================================================================
# 5. STRATIFIED BY Tc RANGE
# =============================================================================
bins = [
    (0,   10,  '< 10 K'),
    (10,  30,  '10–30 K'),
    (30,  77,  '30–77 K'),
    (77,  135, '77–135 K'),
    (135, 999, '> 135 K'),
]

print(f"\n{'Bin':<12}{'n':>6}{'RMSE(K)':>9}{'MAE(K)':>9}"
      f"{'Bias(K)':>9}{'<5K%':>8}{'<10K%':>8}")
print("-" * 63)

results = []
for lo, hi, lbl in bins:
    mask = (y >= lo) & (y < hi)
    n    = mask.sum()
    if n == 0:
        print(f"{lbl:<12}{0:>6}")
        continue
    yt   = y[mask]
    yp   = oof_pred[mask]
    r    = residuals[mask]
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae  = mean_absolute_error(yt, yp)
    bias = r.mean()
    p5   = (np.abs(r) < 5).mean()  * 100
    p10  = (np.abs(r) < 10).mean() * 100
    # R² only reported for bins with sufficient range
    r2b  = r2_score(yt, yp) if yt.std() > 2 else float('nan')

    results.append({
        'Bin': lbl, 'n': n, 'R2': round(r2b, 4),
        'RMSE_K': round(rmse, 3), 'MAE_K': round(mae, 3),
        'Bias_K': round(bias, 3),
        'Within5K_pct': round(p5, 1),
        'Within10K_pct': round(p10, 1),
    })
    r2_str = f"{r2b:.3f}" if not np.isnan(r2b) else "  n/a"
    print(f"{lbl:<12}{n:>6}{rmse:>9.3f}{mae:>9.3f}{bias:>+9.3f}"
          f"{p5:>7.1f}%{p10:>7.1f}%  R²={r2_str}")

print("-" * 63)
print(f"{'OVERALL':<12}{len(y):>6}{rmse_overall:>9.3f}{mae_overall:>9.3f}"
      f"{residuals.mean():>+9.3f}  R²={r2_overall:.4f}")

# =============================================================================
# 6. SAVE RESULTS CSV
# =============================================================================
results_df = pd.DataFrame(results)
results_df.to_csv('tc_stratified_results.csv', index=False)
print("\nSaved -> tc_stratified_results.csv")
